# ERK-KTR Full FOV Stimulation Pipeline - v2 (grid FOV finder)

Same multi-phase ERK-KTR optogenetic full-FOV stimulation experiment as
`stim_rtmsequence.ipynb`, with one difference: the FOV finder.

| | v1 (`stim_rtmsequence.ipynb`) | v2 (this notebook) |
|---|---|---|
| finder | `FOVFinderAgent` | `GridFOVFinderAgent` |
| how it scans | scattered random candidates per well | a contiguous `GRID_ROWS x GRID_COLS` grid per well |
| how it picks | farthest-point spread of candidates passing a CNR gate | density windows on the reconstructed cell cloud, recentred onto the cells |
| ERK selection | `FOVCondition("cnr", ...)` on the candidates | the **same** `FOVCondition`, now gating the density windows |

Both finders satisfy the same `FOVFinder` protocol (`run() -> list[FovPosition]`,
position names `"<well>_<NNNN>"`), so everything downstream is identical -
patterns, pipeline, the batched run loop, and post-processing.
You configure **one** finder object and hand it over with `finder=`; it runs
**during** the experiment (re-scoped per batch), not only up front. The run is
launched with `run_well_patterns_async(ctrl, mic, patterns, finder=finder, ...)`
- a one-line wrapper over `ctrl.run_orchestrator_async(...)` so you don't repeat
`ctrl`/`mic` or pass the orchestrator function by hand.

This v2 keeps the v1 ERK readiness gate. Two independent selectors stack:
**density** (cell count + clumping) is computed *geometrically* by the grid
finder from the reconstructed cell cloud, and the **ERK gate** is a per-cell
`FOVCondition("cnr", ...)` evaluated via `FE_ErkKtr`. So a kept FOV is both a
good-density, not-clumped field **and** mostly ERK-ready (CNR below 1.0 in
>= 70% of cells). To select on density alone, drop `feature_extractor` /
`fov_conditions` from the `GridFOVFinderAgent(...)` call below (density still
works -- it does not depend on the extractor). To go back to v1's finder
entirely, build `finder=FOVFinderAgent(...)` instead.

(Note: you do *not* need `SpatialFE`/an `nn_dist` condition here -- clumping is
already handled by the geometric density knobs. `SpatialFE` is for the old
`FOVFinderAgent`, whose scan only counts cells.)


In [5]:
import os
import time
import pandas as pd
from faro.core.data_structures import (
    Channel,  # basic imaging channel (config, exposure, group)
    PowerChannel,  # imaging channel with light-source power control (adds power 0-100)
    RTMSequence,  # defines one phase of the acquisition (time plan, channels, stim, etc.)
    SegmentationMethod,
    combine,  # compose multi-phase experiments along an axis (t or p)
)

from faro.agents import (
    FOVFinderAgent,
    GridFOVFinderAgent,  # v2: density-aware grid overview finder
    ComposedAgent,
    FOVCondition,
    FOVConditionMonitorAgent,
    WellPattern,  # pairs a well with a build_sequence(fovs) -> RTMSequence callback
    resolve_well_patterns,  # find ALL FOVs up front, combine + batch into one run
    run_well_patterns,  # find + run in sequential batches (6 wells at a time)
    run_well_patterns_async,  # launch run_well_patterns on a worker thread
)


import faro.core.utils as utils

In [6]:
patterns_to_test = pd.read_csv("mixed_normal_patterns.csv")
patterns_to_test["value"] = patterns_to_test["value"].round(0).astype(int)
patterns_to_test["uid"] = patterns_to_test["uid"].astype(int)
patterns_to_test["time"] = patterns_to_test["time"].astype(int)

### Experimental Settings

In [7]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup(
    "TTL_ERK"
)  # select the channel group configured in Micro-Manager

OSError: Line 72: Property,Core,Initialize,1
Error in device "Andor sCMOS Camera": Unable to communicate with the device. (35)



In [ ]:
WELLS = []
START_COL = 2
END_COL = 12
for i, row in enumerate("ABCDEFGH"):
    cols = (
        range(START_COL, END_COL + 1)
        if i % 2 == 0
        else range(END_COL, START_COL - 1, -1)
    )
    WELLS.extend(f"{row}{c}" for c in cols)

In [14]:
## Configuration
SLEEP_BEFORE_EXPERIMENT_START_in_H = (
    0  # delay before acquisition (hours); 0 = start immediately
)

## Storage -- all output (zarr, tracks, TIFFs) goes under this directory
base_path = "E:\\Alex"
experiment_name = "2026-06-26_FreePatternStim_Jungfrau_v2"
path = os.path.join(base_path, experiment_name)

## Stimulation channel -- light used for optogenetic activation
stim_channel = PowerChannel(
    config="CyanStim",  # Micro-Manager channel preset name
    exposure=100,  # stimulation pulse duration (ms)
    group="TTL_ERK",  # Micro-Manager channel group
    power=10,  # light source intensity (0-100)
)

## Imaging channels -- acquired at every timepoint; order matters (channel 0 is used for segmentation)
imaging_channels = (
    PowerChannel(
        config="miRFP", exposure=150, group="TTL_ERK", power=95
    ),  # nuclear marker
    PowerChannel(
        config="mScarlet3", exposure=150, group="TTL_ERK", power=95
    ),  # ERK-KTR reporter
)

## Optocheck channel -- reference channel to verify optogenetic tool expression (longer exposure)
optocheck_channel = PowerChannel(
    config="mCitrine", exposure=600, group="TTL_ERK", power=95
)

### Pipeline Setup

In [15]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    SegmentationMethod(
        name="labels",  # label layer name in the zarr store
        segmentation_class=CellposeV4(
            custom_model_path=r"E:\models\cellpose\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        save_tracked=True,  # persist tracked label masks alongside raw segmentation
    )
]

stimulator = StimWholeFOV()  # illuminate entire FOV (no DMD patterning)
feature_extractor = FE_ErkKtr("labels")  # cytoplasmic/nuclear ratio using "labels" mask
tracker = TrackerTrackpy(search_range=50)  # max 50 px displacement between frames
optocheck = OptoCheckFE(used_mask="labels")  # measure optogenetic reporter per cell

from faro.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,  # runs only on ref_frames (optocheck timepoints)
)

from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(
    storage_path=path,
)  # saves images + stim readout into OME-Zarr

Directory E:\Alex\2026-06-26_FreePatternStim_Jungfrau_v2\tracks already exists


In [10]:
# =============================================================================
# Stimulation patterns  +  per-well GRID FOV-finder configuration
# =============================================================================
PLATE_CALIBRATION_PATH = (
    r".\calib_plate_96.json"  # <-- WellPlatePlan JSON from the MDA plate widget
)

# Grid-FOV-finder settings -----------------------------------------------------
# Tune the density knobs offline first in
# experiments/32_fov_finder/grid_density_fov_finder_tuning.ipynb, then paste the
# same numbers here.
FOVS_PER_WELL = 3  # FOVs kept per well
GRID_ROWS = 5  # overview-scan grid: rows x cols of camera tiles
GRID_COLS = 5  #   (25 tiles imaged per well during the scan)
GRID_OVERLAP = 0.05  # 5% tile overlap so there are no blind seams
FOV_MIN_CELLS = 25  # reject FOV windows with fewer cells
FOV_MAX_CELLS = 125  # reject overly-confluent FOV windows
CLUMP_DISTANCE_UM = 30.0  # a cell with a neighbour within this is "clumped"
MAX_CLUMPED_FRACTION = 0.4  # reject windows with > this fraction clumped
PIXEL_SIZE_UM = 0.65  # mmc.getPixelSizeUm(); None reads it at run time
FLIP_X, FLIP_Y = False, True  # camera->stage axis flips (~180 deg plate rotation);
#   measure with section 8 of grid_fov_finder_demo
TIME_PER_FOV = 3.3  # seconds to image one FOV (used for stim batching)
N_PARALLEL_FOVS = 18  # FOVs imaged per batch (e.g. 6 wells x 3 FOVs)
TIME_BETWEEN_TIMESTEPS = 60.0  # seconds between timepoints

# ERK readiness gate (same as v1): accept a window only where the ERK-KTR signal
# is usable -- CNR below 1.0 in >= 70% of its cells. This is ORTHOGONAL to the
# geometric density band above: the grid finder runs the feature extractor on
# every tile, joins per-cell CNR onto the centroid cloud, and rejects windows
# that pass density but fail this biology gate. (Clumping is NOT done here --
# it's geometric; SpatialFE/nn_dist conditions belong to the old FOVFinderAgent.)
# Drop the feature_extractor / fov_conditions args from GridFOVFinderAgent(...)
# below to select on geometric cell density alone.
ready = FOVCondition("cnr", "below", 1.0, min_fraction=0.65)


# --- Map one CSV waveform (uid) -> a per-well RTMSequence pattern ------------
# A frame is stimulated where the waveform value > threshold, AND that frame's
# `value` sets its stim pulse duration (stim_exposure, in ms). The FOVs are
# filled in at runtime, so build_sequence() takes them as input.
def stim_pattern_from_waveform(
    well, times, values, *, threshold=0.0, rtm_metadata=None
):
    """Build a WellPattern from a per-frame stim waveform.

    A frame ``t`` is a stim frame iff ``value[t] > threshold``, and that frame's
    pulse duration is ``value[t]`` ms (``stim_exposure``). RTMSequence maps
    ``stim_exposure[i]`` to the i-th frame of ``sorted(stim_frames)``, so both
    are built in sorted order.
    """
    val_by_t = {int(t): float(v) for t, v in zip(times, values)}
    stim_frames_sorted = sorted(t for t, v in val_by_t.items() if v > threshold)
    stim_frames = frozenset(stim_frames_sorted)
    # Per-frame pulse duration (ms), aligned to sorted(stim_frames).
    stim_exposure = [val_by_t[t] for t in stim_frames_sorted] or None
    n_frames = int(max(times)) + 1
    base_meta = dict(rtm_metadata or {})

    def build(fovs):  # fovs: list[FovPosition] located at runtime
        return RTMSequence(
            time_plan={"interval": TIME_BETWEEN_TIMESTEPS, "loops": n_frames},
            stage_positions=fovs,
            channels=imaging_channels,
            stim_channels=(stim_channel,),
            stim_frames=stim_frames,
            stim_exposure=stim_exposure,  # value -> per-frame pulse duration (ms)
            ref_channels=(optocheck_channel,),
            ref_frames=frozenset({n_frames - 1}),  # optocheck on the last frame
            rtm_metadata={"well": well, **base_meta},
        )

    return WellPattern(well=well, build_sequence=build)


# One pattern per uid -> one well (in WELLS order) ---------------------------
uids = sorted(patterns_to_test["uid"].unique())
pattern_wells = WELLS[: len(uids)]
assert len(WELLS) >= len(
    uids
), f"Need {len(uids)} wells for {len(uids)} patterns; only {len(WELLS)} defined."

patterns = []
for uid, well in zip(uids, pattern_wells):
    g = patterns_to_test[patterns_to_test.uid == uid].sort_values("time")
    patterns.append(
        stim_pattern_from_waveform(
            well=well,
            times=g["time"].to_numpy(),
            values=g["value"].to_numpy(),  # value > 0 -> stim frame; value = pulse ms
            rtm_metadata={
                "uid": int(uid),
                "treatment_name": f"pattern_{uid}",
            },
        )
    )

# Build ONE configured grid finder and hand it to the run with `finder=`.
# `run_well_patterns` re-scopes it to a single well per batch (a cheap copy that
# shares the loaded plate plan / Cellpose model / extractor), so `wells` and
# `wells_per_phase` here are placeholders -- they are overridden per well.
# Every other knob below (grid, density band, ERK gate, flips) is taken as-is.
# This is the same object you would build and try in grid_fov_finder_demo.ipynb.
finder = GridFOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=pattern_wells,  # placeholder: managed/overridden per well by the run
    fovs_per_well=FOVS_PER_WELL,
    grid_rows=GRID_ROWS,
    grid_cols=GRID_COLS,
    grid_overlap=GRID_OVERLAP,
    min_cells=FOV_MIN_CELLS,
    max_cells=FOV_MAX_CELLS,
    clump_distance_um=CLUMP_DISTANCE_UM,
    max_clumped_fraction=MAX_CLUMPED_FRACTION,
    recenter=True,  # mean-shift each FOV onto its cell cluster
    imaging_channels=imaging_channels,
    segmentator=segmentators[0].segmentation_class,  # reuse the loaded Cellpose model
    seg_channel_index=0,  # segment the nuclear (miRFP) channel
    feature_extractor=FE_ErkKtr("labels"),  # per-cell CNR for the ERK gate (uses ch 1)
    fov_conditions=[ready],  # select FOVs that are ERK-ready (see `ready` above)
    pixel_size_um=PIXEL_SIZE_UM,
    fov_size_um=None,  # derive from pixel size x camera image dims
    flip_x=FLIP_X,
    flip_y=FLIP_Y,
    z=None,  # PFS holds focus
    strict_count=True,  # always return fovs_per_well per well
)

print(f"{len(patterns)} patterns queued -> wells {pattern_wells}")

88 patterns queued -> wells ['A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'B12', 'B11', 'B10', 'B9', 'B8', 'B7', 'B6', 'B5', 'B4', 'B3', 'B2', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'D12', 'D11', 'D10', 'D9', 'D8', 'D7', 'D6', 'D5', 'D4', 'D3', 'D2', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'F12', 'F11', 'F10', 'F9', 'F8', 'F7', 'F6', 'F5', 'F4', 'F3', 'F2', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12', 'H12', 'H11', 'H10', 'H9', 'H8', 'H7', 'H6', 'H5', 'H4', 'H3', 'H2']


In [11]:
patterns[0].build_sequence([])  # test build_sequence() with empty FOVs

RTMSequence(channels=(Channel(config='miRFP', group='TTL_ERK', exposure=150.0), Channel(config='mScarlet3', group='TTL_ERK', exposure=150.0)), time_plan=TIntervalLoops(interval=datetime.timedelta(seconds=60), loops=91), stim_channels=(PowerChannel(config='CyanStim', exposure=100, group='TTL_ERK', power=10),), stim_frames=frozenset({10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68}), stim_exposure=[53.0, 63.0, 74.0, 86.0, 98.0, 110.0, 123.0, 134.0, 146.0, 156.0, 164.0, 172.0, 177.0, 181.0, 183.0, 183.0, 182.0, 178.0, 174.0, 167.0, 160.0, 152.0, 143.0, 134.0, 124.0, 115.0, 105.0, 96.0, 87.0, 79.0], ref_channels=(PowerChannel(config='mCitrine', exposure=600, group='TTL_ERK', power=95),), ref_frames=frozenset({90}), rtm_metadata={'well': 'A2', 'uid': 0, 'treatment_name': 'pattern_0'})

### GUI (optional)

Opens a napari viewer with the Micro-Manager widget for live camera view and manual focusing. FOV positions are **not** picked here — they are located automatically per well by the FOV finder during the run.

In [9]:
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(
    viewer, mmcore=mic.mmc
)  # widget to control Micro-Manager from napari
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

### Run — find FOVs and stimulate, one batch of wells at a time (mode B)

`run_well_patterns` plugs the FOV finder into the experiment **during** the run:

1. Scan `WELLS_PER_BATCH` wells (each as a `GRID_ROWS x GRID_COLS` grid -> density+ERK picks `WELLS_PER_BATCH x FOVS_PER_WELL` FOVs).
2. Build those FOVs' RTM events **on the fly** and run that batch's time-lapse to completion.
3. Scan + run the next batch, and so on.

Because each batch's events are generated *after* its scan finishes (and run via `continue_experiment`, which restarts the per-batch time-lapse clock from 0), **the finder's scan time never corrupts the time-lapse schedule**. Every FOV is a physically distinct position, so all batches accumulate into a single store / one tracks DataFrame — no per-phase bookkeeping.

It's launched via `ctrl.run_orchestrator_async(...)`, which runs the (blocking) batch loop on a controller worker thread and returns an **`OrchestratorHandle`** immediately, so napari stays responsive. The handle gives two levels of progress through one object:

- **orchestrator level** — `run_handle.status()` → `state`, `step`/`n_steps`, `message` (e.g. `"batch 3/15: C2, C3, …"`);
- **drill-down** — `run_handle.current_run` is the `RunHandle` of the batch currently acquiring (`status().n_events_acquired / n_events_total`, `current_fov`); `currentRunChanged` fires when it advances to the next batch.

`run_handle.cancel()` aborts the in-flight batch and halts between batches; `run_handle.wait()` blocks until done. This is the **same async/cancel/drill-down contract every multi-run agent will use** — e.g. `ctrl.run_orchestrator_async(composed_agent.run)` for a BO experiment.

> **Mode A alternative** (all FOVs found up front, one combined acquisition): `events = resolve_well_patterns(mic, patterns, finder=finder, time_per_fov=TIME_PER_FOV, n_parallel=N_PARALLEL_FOVS).events` then `ctrl.run_experiment(events)`. Use this only when the up-front scan of every well is acceptable.

In [10]:
# Pre-flight: validate a representative pattern's events before committing to the run.
_probe_ctrl = Controller(mic, pipeline, writer=None)  # no writer; validation only
from faro.core.utils import FovPosition

_probe_fovs = [FovPosition(x=0.0, y=0.0, z=None, name="A2_0000")]
_probe_events = list(patterns[0].build_sequence(_probe_fovs))
ok = _probe_ctrl.validate_events(_probe_events)
print("pattern[0] events valid:", ok, "| n events:", len(_probe_events))
# Also eyeball the per-frame stim exposures that will actually fire:
print(
    "stim exposures (ms):",
    sorted({c.exposure for e in _probe_events for c in e.stim_channels}),
)

pattern[0] events valid: True | n events: 91
stim exposures (ms): [53.0, 63.0, 74.0, 79.0, 86.0, 87.0, 96.0, 98.0, 105.0, 110.0, 115.0, 123.0, 124.0, 134.0, 143.0, 146.0, 152.0, 156.0, 160.0, 164.0, 167.0, 172.0, 174.0, 177.0, 178.0, 181.0, 182.0, 183.0]


In [ ]:
WELLS_PER_BATCH = 6  # wells found + run together (6 x 3 FOVs = 18 FOVs per batch)

ctrl = Controller(mic, pipeline, writer=writer)

# Live status + pause/stop buttons (re-binds when each batch's run starts).
from faro.widgets import ExperimentStatusWidget

viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)

# Optional delay before the first batch (set SLEEP_BEFORE_EXPERIMENT_START_in_H above).
for _ in range(int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

# Launch the batch-sequential run on a controller worker thread; returns at once
# so napari stays responsive. run_orchestrator_async injects the OrchestratorHandle
# as `progress`, so run_well_patterns reports "batch i/n" and is cancellable.
#   run_handle.status()           -> orchestrator level (state, step/n_steps, message)
#   run_handle.current_run        -> the RunHandle of the batch currently acquiring
#   run_handle.cancel()           -> abort the in-flight batch + stop between batches
#   run_handle.wait()             -> block until all batches finish
# One configured grid finder, launched on a worker thread. run_well_patterns_async
# wraps ctrl.run_orchestrator_async (so napari stays responsive and the run is
# cancellable) without re-passing ctrl/mic or the orchestrator function.
run_handle = run_well_patterns_async(
    ctrl,
    mic,
    patterns,
    wells_per_batch=WELLS_PER_BATCH,
    finder=finder,  # v2: the grid finder built above (density + ERK gate)
    time_per_fov=TIME_PER_FOV,
    n_parallel=N_PARALLEL_FOVS,
    stim_mode="current",
    finish=True,  # close the store after the last batch
)
print("Experiment launched. run_handle.status().state ->", run_handle.status().state)

In [11]:
run_handle.cancel()  # cancel the run (if needed) -- stops the current batch + prevents future batches

### Monitor / cancel (optional)

Re-run the cell below any time to snapshot progress without blocking. Cancel the whole run with `run_handle.cancel()`.

In [ ]:
# Snapshot progress without blocking (re-run anytime while the experiment runs).
# The ExperimentStatusWidget already renders the current batch's per-event detail;
# this prints the orchestrator-level view + a drill-down into the live batch.
s = run_handle.status()
print(f"orchestrator: {s.state}" + (f"  |  {s.message}" if s.message else ""))

cur = run_handle.current_run
if cur is not None:
    cs = cur.status()
    print(
        f"  current batch: {cs.state}  "
        f"{cs.n_events_acquired}/{cs.n_events_total} events"
        + (f", FOV {cs.current_fov}" if cs.current_fov is not None else "")
    )

# Stop early (aborts the in-flight batch, then halts between batches):
#   run_handle.cancel()

### Post-processing

`run_handle.wait()` blocks until every batch has finished (`run_well_patterns(finish=True)`
already flushed the pipeline and closed the zarr store). Then we merge the per-FOV
track parquet files into a single `exp_data.parquet`.

In [12]:
run_handle.wait()  # block until all batches finish (run already closed the store)

utils.generate_exp_data_from_tracks(path)  # merge per-FOV tracks into exp_data.parquet

In [14]:
pd.read_parquet(os.path.join(path, "exp_data.parquet"))

,label,x,y,well,uid,treatment_name,fov,timestep,fname,time,...,mean_intensity_C0_ring,mean_intensity_C1_ring,median_intensity_C0_ring,median_intensity_C1_ring,cnr_mean,cnr,stim_power,stim_exposure,ref_mean_intensity,time_offset
0,1,24.079023,762.409483,A2,0,pattern_0,0,0,000_00000,0.000000,...,266.244240,1755.576037,262.0,1671.0,1.004681,0.969539,NaN,NaN,NaN,NaN
1,2,115.262712,664.752119,A2,0,pattern_0,0,0,000_00000,0.000000,...,294.264865,1611.097297,289.0,1627.0,0.783638,0.791535,NaN,NaN,NaN,NaN
2,3,232.250340,644.259864,A2,0,pattern_0,0,0,000_00000,0.000000,...,305.343750,2328.839286,303.5,2352.5,0.956920,0.984310,NaN,NaN,NaN,NaN
3,4,297.828194,186.046256,A2,0,pattern_0,0,0,000_00000,0.000000,...,295.718232,2777.309392,294.0,2669.0,0.918091,0.889963,NaN,NaN,NaN,NaN
4,5,315.103960,34.371287,A2,0,pattern_0,0,0,000_00000,0.000000,...,272.787356,1332.327586,273.5,1310.0,0.946463,0.938395,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9018,98,845.651226,255.059946,A5,3,pattern_3,9,90,009_00090,5429.700195,...,277.022727,7636.645455,275.0,7769.5,1.088046,1.114146,NaN,NaN,2299.754768,NaN
9019,99,873.545939,309.849534,A5,3,pattern_3,9,90,009_00090,5429.700195,...,280.504274,2468.282051,280.0,2389.5,0.735560,0.718863,NaN,NaN,3669.704394,NaN
9020,100,907.555556,202.073552,A5,3,pattern_3,9,90,009_00090,5429.700195,...,259.535211,1744.380282,259.0,1734.0,0.768018,0.769299,NaN,NaN,1247.978091,NaN
9021,101,951.501471,448.010294,A5,3,pattern_3,9,90,009_00090,5429.700195,...,267.648148,1346.162037,268.5,1309.5,0.895484,0.866071,NaN,NaN,1406.601471,NaN
